<a href="https://colab.research.google.com/github/NaghamZidiah/FlyRank-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I chose **K-Means clustering** because my ML-03 lane is Structured Content Archetype Clustering.

The goal is to group content pages with similar observed performance patterns without using a predefined target label.

K-Means fits this task because it can group pages based on multiple numerical performance signals and produce interpretable cluster profiles.

For the modeling step, I will compare the clustering model with the simple clustering baseline created in ML-07. The baseline uses three GSC performance features: impressions, clicks, and average position.

The model will be evaluated using the **silhouette score**, where a higher score indicates better separation between clusters.

The model comparison will use the same data and evaluation metric so that the comparison with the baseline is fair.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
print("Token loaded successfully!")

con = duckdb.connect()

con.execute(f"""
CREATE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("Connected successfully!")

rel = "hf://datasets/FlyRank/internship-warehouse"

Token loaded successfully!
Connected successfully!


In [ ]:
# Load and prepare the clustering sample for ML-08

import numpy as np
import pandas as pd

cluster_query = f"""
SELECT
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_impressions > 0
  AND gsc_avg_position IS NOT NULL
"""

cluster_df = con.sql(cluster_query).df()

cluster_df = cluster_df.replace(
    [np.inf, -np.inf],
    np.nan
).dropna()

# Use the same reproducible sample size as the ML-07 baseline
model_sample = cluster_df.sample(
    n=min(100000, len(cluster_df)),
    random_state=42
).reset_index(drop=True)

# Create the transformed features for the ML-08 model
model_sample["log_impressions"] = np.log1p(
    model_sample["gsc_impressions"]
)

model_sample["log_clicks"] = np.log1p(
    model_sample["gsc_clicks"]
)

print("Clustering sample size:", len(model_sample))
print("Columns:", model_sample.columns.tolist())

display(model_sample.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Clustering sample size: 100000
Columns: ['gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'log_impressions', 'log_clicks']


,gsc_impressions,gsc_clicks,gsc_avg_position,log_impressions,log_clicks
0,117,1,23.461538,4.770685,0.693147
1,21,0,3.714286,3.091042,0.000000
2,125,1,11.064000,4.836282,0.693147
3,16,0,47.000000,2.833213,0.000000
4,35,0,9.657143,3.583519,0.000000


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


Because this is an unsupervised clustering task, there is no target label to split by.

To make the comparison fair, I use the same reproducible development sample for both the baseline and the model. I split this sample into training and test sets using a fixed random state.

The clustering models are fitted on the training set only. The test set is then used to evaluate cluster separation using the silhouette score.

This prevents the evaluation data from influencing the model fitting and allows a direct baseline-versus-model comparison on the same held-out data.

I use a 80/20 train-test split with `random_state=42` for reproducibility.

In [ ]:
# Split design for fair baseline vs model comparison

from sklearn.model_selection import train_test_split

# Use the same 100,000-row clustering sample for both baseline and model
train_cluster, test_cluster = train_test_split(
    model_sample,
    test_size=0.20,
    random_state=42
)

print("Total clustering sample:", len(model_sample))
print("Training rows:", len(train_cluster))
print("Test rows:", len(test_cluster))

Total clustering sample: 100000
Training rows: 80000
Test rows: 20000


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I first fit the clustering baseline using the original GSC features: impressions, clicks, and average position.

I then test K-Means models using log-transformed impressions and clicks together with average position. The log transformation reduces the effect of highly skewed count variables.

For a fair comparison, both the baseline and the final model are fitted on the same 80% training split and evaluated on the same 20% test split using the silhouette score.

I compare several values of k for the model and select the best one based on the training silhouette score.

In [ ]:
# Baseline K-Means model

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

baseline_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position"
]

# Prepare training and test features
X_train_baseline = train_cluster[baseline_features].copy()
X_test_baseline = test_cluster[baseline_features].copy()

# Scale the baseline features
baseline_scaler = StandardScaler()

X_train_baseline_scaled = baseline_scaler.fit_transform(
    X_train_baseline
)

X_test_baseline_scaled = baseline_scaler.transform(
    X_test_baseline
)

# Fit the baseline with 3 clusters
baseline_model = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

baseline_train_labels = baseline_model.fit_predict(
    X_train_baseline_scaled
)

baseline_test_labels = baseline_model.predict(
    X_test_baseline_scaled
)

# Evaluate on the held-out test set
baseline_test_silhouette = silhouette_score(
    X_test_baseline_scaled,
    baseline_test_labels
)

print("Baseline test silhouette score:",
      round(baseline_test_silhouette, 4))

print("\nBaseline test cluster sizes:")
print(
    pd.Series(baseline_test_labels)
    .value_counts()
    .sort_index()
)

Baseline test silhouette score: 0.613

Baseline test cluster sizes:
0     3139
1    16861
Name: count, dtype: int64


In [ ]:
print("Unique baseline test clusters:")
print(sorted(set(baseline_test_labels)))

print("\nNumber of unique clusters:")
print(len(set(baseline_test_labels)))

print("\nTest cluster counts:")
print(
    pd.Series(baseline_test_labels)
    .value_counts()
    .sort_index()
)

Unique baseline test clusters:
[np.int32(0), np.int32(1)]

Number of unique clusters:
2

Test cluster counts:
0     3139
1    16861
Name: count, dtype: int64


In [ ]:
print("Training cluster counts:")
print(
    pd.Series(baseline_train_labels)
    .value_counts()
    .sort_index()
)

Training cluster counts:
0    12295
1    67699
2        6
Name: count, dtype: int64


### Baseline cluster distribution check

The baseline K-Means model was fitted with 3 clusters on the 80% training split.

The training set contains all three clusters, but Cluster 2 contains only 6 observations. When the fitted model is applied to the 20% test split, no observations are assigned to Cluster 2.

This indicates that the third cluster represents a very small group of extreme observations and does not generalize to the held-out test sample.

The test silhouette score of 0.613 is therefore based on the two clusters represented in the test set. This is not treated as evidence that the third cluster is a robust content archetype.

In [ ]:
# Test different numbers of clusters for the ML model

model_features = [
    "log_impressions",
    "log_clicks",
    "gsc_avg_position"
]

X_train_model = train_cluster[model_features].copy()
X_test_model = test_cluster[model_features].copy()

# Scale the transformed features
model_scaler = StandardScaler()

X_train_model_scaled = model_scaler.fit_transform(
    X_train_model
)

X_test_model_scaled = model_scaler.transform(
    X_test_model
)

# Compare k values from 2 to 6
k_results = []

for k in range(2, 7):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    train_labels = kmeans.fit_predict(
        X_train_model_scaled
    )

    train_silhouette = silhouette_score(
        X_train_model_scaled,
        train_labels
    )

    k_results.append({
        "k": k,
        "train_silhouette": train_silhouette
    })

k_results_df = pd.DataFrame(k_results)

display(
    k_results_df.round(4)
)

,k,train_silhouette
0,2,0.5378
1,3,0.5166
2,4,0.4562
3,5,0.4622
4,6,0.4589


### Selecting the number of clusters

I compared K-Means models with different numbers of clusters using the training silhouette score.

Among the tested values, **k=2 achieved the highest training silhouette score (0.5378)**. The scores decreased as the number of clusters increased, with k=4, k=5, and k=6 producing lower separation.

The k=3 solution also produced a very small cluster in the training data, which did not appear in the test set. Therefore, I will evaluate the candidate values on the held-out test set before selecting the final number of clusters.

In [ ]:
# Compare the ML-08 model using the same k as the baseline

model_k3 = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

model_k3_train_labels = model_k3.fit_predict(
    X_train_model_scaled
)

model_k3_test_labels = model_k3.predict(
    X_test_model_scaled
)

model_k3_test_silhouette = silhouette_score(
    X_test_model_scaled,
    model_k3_test_labels
)

print("ML-08 model with k=3")
print(
    "Test silhouette score:",
    round(model_k3_test_silhouette, 4)
)

print("\nTest cluster sizes:")
print(
    pd.Series(model_k3_test_labels)
    .value_counts()
    .sort_index()
)

ML-08 model with k=3
Test silhouette score: 0.5196

Test cluster sizes:
0     2743
1     2289
2    14968
Name: count, dtype: int64


In [ ]:
# Select the best k based on training silhouette score

best_k = int(
    k_results_df.loc[
        k_results_df["train_silhouette"].idxmax(),
        "k"
    ]
)

best_train_silhouette = k_results_df[
    k_results_df["k"] == best_k
]["train_silhouette"].iloc[0]

print("Best number of clusters:", best_k)
print(
    "Best training silhouette score:",
    round(best_train_silhouette, 4)
)

# Fit the final model using the selected k
final_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=10
)

final_train_labels = final_model.fit_predict(
    X_train_model_scaled
)

final_test_labels = final_model.predict(
    X_test_model_scaled
)

# Evaluate the final model on the held-out test set
final_test_silhouette = silhouette_score(
    X_test_model_scaled,
    final_test_labels
)

print(
    "\nFinal model test silhouette score:",
    round(final_test_silhouette, 4)
)

print("\nFinal model test cluster sizes:")
print(
    pd.Series(final_test_labels)
    .value_counts()
    .sort_index()
)

Best number of clusters: 2
Best training silhouette score: 0.5378

Final model test silhouette score: 0.5372

Final model test cluster sizes:
0    17709
1     2291
Name: count, dtype: int64


### Additional k comparison

To check whether the difference from the ML-07 baseline was caused only by using a different number of clusters, I also evaluated the ML-08 feature set with **k=3**, matching the baseline.

The ML-08 model achieved a test silhouette score of **0.5196** with k=3, which was lower than its **0.5372** score with k=2.

Therefore, the lower performance of ML-08 is not explained only by the difference in k. The log-transformed feature representation also did not improve cluster separation compared with the raw-feature baseline.

In [ ]:
# Compare the baseline and final model

comparison_df = pd.DataFrame({
    "method": [
        "ML-07 clustering baseline",
        "ML-08 K-Means model"
    ],
    "features": [
        "raw impressions + clicks + position",
        "log impressions + log clicks + position"
    ],
    "n_clusters": [
        3,
        best_k
    ],
    "test_silhouette": [
        baseline_test_silhouette,
        final_test_silhouette
    ]
})

comparison_df["improvement"] = (
    comparison_df["test_silhouette"]
    - baseline_test_silhouette
)

display(
    comparison_df.round(4)
)

,method,features,n_clusters,test_silhouette,improvement
0,ML-07 clustering baseline,raw impressions + clicks + position,3,0.6130,0.0000
1,ML-08 K-Means model,log impressions + log clicks + position,2,0.5372,-0.0758


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


Because clustering is an unsupervised task, errors are interpreted as cases where the resulting clusters may be small, unstable, or difficult to distinguish as meaningful content archetypes.

I will inspect the final cluster sizes and average feature values to understand what each cluster represents.

The main comparison is between the final ML-08 model and the ML-07 baseline. Since the ML-08 model did not improve the test silhouette score, the analysis focuses on understanding the limitations of the model rather than treating higher complexity as automatically better.

In [ ]:
# Inspect final model cluster sizes and profiles

final_test_analysis = test_cluster.copy()

final_test_analysis["cluster"] = final_test_labels

print("Final model cluster sizes:")
display(
    final_test_analysis["cluster"]
    .value_counts()
    .sort_index()
)

print("\nFinal model cluster profiles:")

display(
    final_test_analysis
    .groupby("cluster")[
        [
            "gsc_impressions",
            "gsc_clicks",
            "gsc_avg_position",
            "log_impressions",
            "log_clicks"
        ]
    ]
    .mean()
    .round(3)
)

Final model cluster sizes:


,count
cluster,
0,17709
1,2291



Final model cluster profiles:


,gsc_impressions,gsc_clicks,gsc_avg_position,log_impressions,log_clicks
cluster,,,,,
0,45.240,0.003,16.848,2.703,0.002
1,326.624,2.019,8.805,5.106,0.961


### Final cluster interpretation

The final ML-08 model produced two clusters on the held-out test set.

**Cluster 0** contained 17,709 observations and had lower average impressions (45.240), very low average clicks (0.003), and a weaker average position (16.848).

**Cluster 1** contained 2,291 observations and showed higher average impressions (326.624), higher average clicks (2.019), and a better average position (8.805).

These profiles suggest that the model mainly separates observations into a larger lower-performance group and a smaller higher-performance group.

However, these clusters should be interpreted as **performance-based archetypes**, not content types, because the clustering model used only numerical GSC performance features. The model did not improve the baseline silhouette score, so these clusters are useful for interpretation but are not treated as a superior replacement for the ML-07 baseline.

### Limitations and potential errors

The main limitation is that the clustering model simplifies performance into only two groups. This may hide meaningful variation among individual content pages.

The earlier k=3 baseline also produced a very small cluster, showing that extreme high-performance observations can form unstable groups. In the final k=2 model, the clusters are more stable in the test set, but the model still captures broad performance differences rather than distinct content archetypes.

The lower test silhouette score compared with the ML-07 baseline indicates that the log-transformed features did not provide better cluster separation for this dataset.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.